In [1]:
import sys
from pathlib import Path
import wfdb

In [2]:
import pandas as pd

In [4]:
database_name = 'mimic3wdb/1.0'

In [ ]:
# each subject may be associated with multiple records
subjects = wfdb.get_record_list(database_name)
print(f"The '{database_name}' database contains data from {len(subjects)} subjects")

The 'mimic3wdb/1.0' database contains data from 67830 subjects


In [6]:
subjects[0:5]

['30/3000003/', '30/3000031/', '30/3000051/', '30/3000060/', '30/3000063/']

In [15]:
records_csv = pd.read_csv("matching_records.csv")
last_subject = records_csv.iloc[-1].dir.split("1.0/")
last_subject = last_subject[-1]
print(last_subject)
last_subject_idx = subjects.index(last_subject+'/')
print(last_subject_idx)

30/3000714
40


In [52]:
# iterate the subjects to get a list of records
max_records_to_load = 800
start = 0

records = []
for subject in subjects[start:-1]:
    studies = wfdb.get_record_list(f'{database_name}/{subject}')
    for study in studies:
        records.append(Path(f'{subject}{study}'))
        # stop if we've loaded enough records
        if len(records) >= max_records_to_load:
            print("Reached maximum required number of records.")
            break
    if len(records) >= max_records_to_load:
        break

print(f"Loaded {len(records)} records from the '{database_name}' database.")

Reached maximum required number of records.
Loaded 800 records from the 'mimic3wdb/1.0' database.


In [53]:
records

[WindowsPath('30/3000003/3000003'),
 WindowsPath('30/3000003/3000003n'),
 WindowsPath('30/3000003/3000003_0001'),
 WindowsPath('30/3000003/3000003_0002'),
 WindowsPath('30/3000003/3000003_0003'),
 WindowsPath('30/3000003/3000003_0004'),
 WindowsPath('30/3000003/3000003_0005'),
 WindowsPath('30/3000003/3000003_0006'),
 WindowsPath('30/3000003/3000003_0007'),
 WindowsPath('30/3000003/3000003_0008'),
 WindowsPath('30/3000003/3000003_0009'),
 WindowsPath('30/3000003/3000003_0010'),
 WindowsPath('30/3000003/3000003_0011'),
 WindowsPath('30/3000003/3000003_0012'),
 WindowsPath('30/3000003/3000003_0013'),
 WindowsPath('30/3000003/3000003_0014'),
 WindowsPath('30/3000003/3000003_0015'),
 WindowsPath('30/3000003/3000003_0016'),
 WindowsPath('30/3000003/3000003_0017'),
 WindowsPath('30/3000031/3000031n'),
 WindowsPath('30/3000051/3000051'),
 WindowsPath('30/3000051/3000051_0001'),
 WindowsPath('30/3000051/3000051_0002'),
 WindowsPath('30/3000051/3000051_0003'),
 WindowsPath('30/3000060/3000060')

In [54]:
# format and print first five records
first_five_records = [str(x) for x in records[0:5]]
first_five_records = "\n - ".join(first_five_records)
print(f"First five records: \n - {first_five_records}")

last_five_records = [str(x) for x in records[-5:]]
last_five_records = "\n - ".join(last_five_records)
print(f"Last five records: \n - {last_five_records}")

print("""
Note the formatting of these records (mimic3wdb/1.0 uses only 2 levels, unlike mimic4wdb):
 - intermediate directory ('30' in this case, based on the first 2 digits of the record id)
 - record/subject id (e.g. '3000003') -- in mimic3wdb there is no separate 'p<subject>' folder,
   the numeric id doubles as both the subject and the record identifier
 """)

First five records: 
 - 30\3000003\3000003
 - 30\3000003\3000003n
 - 30\3000003\3000003_0001
 - 30\3000003\3000003_0002
 - 30\3000003\3000003_0003
Last five records: 
 - 30\3000714\3000714_0035
 - 30\3000714\3000714_0036
 - 30\3000714\3000714_0037
 - 30\3000714\3000714_0038
 - 30\3000715\3000715

Note the formatting of these records (mimic3wdb/1.0 uses only 2 levels, unlike mimic4wdb):
 - intermediate directory ('30' in this case, based on the first 2 digits of the record id)
 - record/subject id (e.g. '3000003') -- in mimic3wdb there is no separate 'p<subject>' folder,
   the numeric id doubles as both the subject and the record identifier
 


In [55]:
# In mimic3wdb/1.0 the flat 'records' list mixes together, for each subject:
#   - the master multi-segment record   (e.g. '3000003')
#   - the numerics record                (e.g. '3000003n')
#   - the individual waveform segments   (e.g. '3000003_0001', '3000003_0002', ...)
# Only the master record is a true multi-segment header (it has .seg_name and works
# with rd_segments=True). So we filter for master records first: their name is purely
# numeric, with no '_' (segment) or trailing 'n' (numerics) suffix.
master_records = [r for r in records if r.name.isdigit()]
print(f"Found {len(master_records)} master (multi-segment) records out of {len(records)} total entries.")

# Specify the 4th master record (note, in Python indexing begins at 0)
idx = 3
record = master_records[idx]
record_dir = f'{database_name}/{record.parent}'
record_dir = record_dir.replace("\\","/")
print("PhysioNet directory specified for record: {}".format(record_dir))

Found 41 master (multi-segment) records out of 800 total entries.
PhysioNet directory specified for record: mimic3wdb/1.0/30/3000063


In [28]:
record_name = record.name  # e.g. '3000003' -- this is the master multi-segment record name
print("Record name: {}".format(record_name))

Record name: 3000717


In [29]:
record_dir

'mimic3wdb/1.0/30/3000717'

In [30]:
# Lê de novo a metadata para obter as informações do segmento
record_data = wfdb.rdheader(record_name, pn_dir=record_dir, rd_segments=True)
remote_url = "https://physionet.org/content/" + record_dir + "/" + record_name + ".hea"
print(f"Done: metadata loaded for record '{record_name}' from the header file at:\n{remote_url}")

Done: metadata loaded for record '3000717' from the header file at:
https://physionet.org/content/mimic3wdb/1.0/30/3000717/3000717.hea


In [31]:
print(f"- Number of signals: {record_data.n_sig}".format())
print(f"- Duration: {record_data.sig_len/(record_data.fs*60*60):.1f} hours") 
print(f"- Base sampling frequency: {record_data.fs} Hz")

- Number of signals: 8
- Duration: 228.4 hours
- Base sampling frequency: 125 Hz


In [32]:
seg_dict = record_data.__dict__
seg_dict

{'record_name': '3000717',
 'n_sig': 8,
 'fs': 125,
 'counter_freq': None,
 'base_counter': None,
 'sig_len': 102780846,
 'base_time': datetime.time(21, 21, 39, 232000),
 'base_date': None,
 'comments': ['Location: micu'],
 'sig_name': ['II', 'aVR', 'ART', 'ABP', 'RESP', 'III', 'PLETH', 'I'],
 'layout': 'variable',
 'segments': [<wfdb.io.record.Record at 0x28dc0b3f810>,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
 'seg_name': ['3000717_layout',
  '3000717_0001',
  '3000717_0002',
  '3000717_0003',
  '3000717_0004',
  '3000717_0005',
  '3000717_0006',
  '3000717_0007',
  '3000717_0008',
  '3000717_0009',
  '3000717_0010',
  '3000717_0011',
  '3000717_0012',
  '3000717_0013',
  '3000717_0014',
  '3000717_0015',
  '3000717_0016',
  '3000717_0017',
  '3000717_0018',
  '3000717_0019',
  '3000717_0020',
  '3000717_0021',
  '3000717_0022',
  '3000717_0023',
  '~',
  '3000717_0024',
  '3000717_0025',
  '3000717_0026',
  '3000717_0027',
  '3000717_0028',
  '3000717_0029',
  '3000717

In [33]:
segments = record_data.seg_name
segments = [seg for seg in segments if seg != '~']
layout_rem = f"{record_name}_layout"
segments.remove(layout_rem)
segments

['3000717_0001',
 '3000717_0002',
 '3000717_0003',
 '3000717_0004',
 '3000717_0005',
 '3000717_0006',
 '3000717_0007',
 '3000717_0008',
 '3000717_0009',
 '3000717_0010',
 '3000717_0011',
 '3000717_0012',
 '3000717_0013',
 '3000717_0014',
 '3000717_0015',
 '3000717_0016',
 '3000717_0017',
 '3000717_0018',
 '3000717_0019',
 '3000717_0020',
 '3000717_0021',
 '3000717_0022',
 '3000717_0023',
 '3000717_0024',
 '3000717_0025',
 '3000717_0026',
 '3000717_0027',
 '3000717_0028',
 '3000717_0029',
 '3000717_0030',
 '3000717_0031',
 '3000717_0032',
 '3000717_0033',
 '3000717_0034',
 '3000717_0035',
 '3000717_0036',
 '3000717_0037',
 '3000717_0038',
 '3000717_0039',
 '3000717_0040',
 '3000717_0041',
 '3000717_0042',
 '3000717_0043',
 '3000717_0044',
 '3000717_0045',
 '3000717_0046',
 '3000717_0047',
 '3000717_0048',
 '3000717_0049',
 '3000717_0050',
 '3000717_0051',
 '3000717_0052',
 '3000717_0053',
 '3000717_0054',
 '3000717_0055',
 '3000717_0056',
 '3000717_0057',
 '3000717_0058',
 '3000717_0059

In [34]:
# segment_data was loaded with rd_segments=True in the cell above, so it's a MultiRecord
# and segment_data.seg_name holds the list of this record's individual segments.
segment_read = segments[5]
segment_metadata = wfdb.rdheader(record_name=segment_read, pn_dir=record_dir)

# NOTE: mimic3wdb/1.0 only has 2 directory levels (intermediate dir + subject/record dir),
# unlike mimic4wdb/0.1.0 which has 3 (waves/p1xx/subject/study). So the subject id here is
# the LAST component of record_dir itself, not its parent.
print(f"""Header metadata loaded for: 
- the segment '{segment_read}'
- in record '{record_name}'
- for subject '{str(Path(record_dir).parts[-1])}'
""")

Header metadata loaded for: 
- the segment '3000717_0006'
- in record '3000717'
- for subject '3000717'



In [35]:
print(f"This segment contains the following signals: {segment_metadata.sig_name}")
print(f"The signals are measured in units of: {segment_metadata.units}")

This segment contains the following signals: ['II', 'ABP']
The signals are measured in units of: ['mV', 'mmHg']


In [36]:
print(f"The signals have a base sampling frequency of {segment_metadata.fs:.1f} Hz")
print(f"and they last for {segment_metadata.sig_len/(segment_metadata.fs*60):.1f} minutes")

The signals have a base sampling frequency of 125.0 Hz
and they last for 215.5 minutes


In [37]:
import pandas as pd
from pprint import pprint

In [38]:
print(f"Earlier, we loaded {len(records)} records from the '{database_name}' database.")

Earlier, we loaded 7200 records from the 'mimic3wdb/1.0' database.


In [39]:
required_sigs = ['ABP', 'ART']

In [40]:
# convert from minutes to seconds
req_seg_duration = 5*60 

In [42]:
# Return the files that contains the serached signals or conditions
matching_recs = {'dir':[], 'seg_name':[], 'length':[]}

# IMPORTANT: only iterate over MASTER (multi-segment) records here.
# rd_segments=True / .seg_name only works on the master record (name is purely
# numeric, e.g. '3000003'); calling it on a numerics record ('3000003n') or an
# individual segment ('3000003_0001') returns a plain Record with no .seg_name,
# which raised the AttributeError above.
print(f"Searching {len(master_records)} master records (out of {len(records)} total entries)...")

for record in master_records:
    print('Record: {}'.format(record), end="", flush=True)
    record_dir = f'{database_name}/{record.parent}'
    record_dir = record_dir.replace("\\","/")
    record_name = record.name
    print(' (reading data)')
    try:
        record_data = wfdb.rdheader(record_name,
                                    pn_dir=record_dir,
                                    rd_segments=True)
    except Exception as e:
        print(f' (skipping, could not read header: {e})')
        continue

    # Check whether the required signals are present in the record
    sigs_present = record_data.sig_name

    # Get the segments for the record
    segments = record_data.seg_name

    # Check to see if the segment is 5 min long
    # If not, move to the next one
    gen = (segment for segment in segments if segment != '~')
    for segment in gen:
        print(' - Segment: {}'.format(segment), end="", flush=True)
        segment_metadata = wfdb.rdheader(record_name=segment,
                                         pn_dir=record_dir)
        seg_length = segment_metadata.sig_len/(segment_metadata.fs)

        if seg_length < req_seg_duration:
            print(f' (too short at {seg_length/60:.1f} mins)')
            continue

        # Next check that all required signals are present in the segment
        sigs_present = segment_metadata.sig_name
        print(f"\nSinais presentes: {sigs_present}")
        
        if any(x in sigs_present for x in required_sigs):
            matching_recs['dir'].append(record_dir)
            matching_recs['seg_name'].append(segment)
            matching_recs['length'].append(seg_length)
            print(' (met requirements)')
        else:
            print(' (long enough, but missing signal(s))')

print(f"A total of {len(matching_recs['dir'])} records met the requirements:")

df_matching_recs = pd.DataFrame(data=matching_recs)
# df_matching_recs.to_csv('matching_records_0_800.csv', index=False)
#p=1

Searching 200 master records (out of 7200 total entries)...
Record: 30\3000714\3000714 (reading data)
 - Segment: 3000714_layout (too short at 0.0 mins)
 - Segment: 3000714_0001
Sinais presentes: ['RESP', 'II', 'V', 'AVR', 'PLETH', 'ABP']
 (met requirements)
 - Segment: 3000714_0002 (too short at 1.7 mins)
 - Segment: 3000714_0003 (too short at 2.9 mins)
 - Segment: 3000714_0004
Sinais presentes: ['RESP', 'II', 'V', 'AVR', 'PLETH', 'ABP']
 (met requirements)
 - Segment: 3000714_0005 (too short at 0.1 mins)
 - Segment: 3000714_0006 (too short at 0.0 mins)
 - Segment: 3000714_0007 (too short at 0.1 mins)
 - Segment: 3000714_0008 (too short at 0.1 mins)
 - Segment: 3000714_0009
Sinais presentes: ['PLETH', 'RESP', 'ABP', 'II', 'V', 'AVR']
 (met requirements)
 - Segment: 3000714_0010
Sinais presentes: ['PLETH', 'RESP', 'ABP', 'II', 'V', 'AVR', 'CVP']
 (met requirements)
 - Segment: 3000714_0011 (too short at 3.4 mins)
 - Segment: 3000714_0012
Sinais presentes: ['PLETH', 'RESP', 'ABP', 'II',

In [43]:
print(f"A total of {len(matching_recs['dir'])} out of {len(records)} records met the requirements.")

relevant_segments_names = "\n - ".join(matching_recs['seg_name'])
print(f"\nThe relevant segment names are:\n - {relevant_segments_names}")

relevant_dirs = "\n - ".join(matching_recs['dir'])
print(f"\nThe corresponding directories are: \n - {relevant_dirs}")

A total of 526 out of 7200 records met the requirements.

The relevant segment names are:
 - 3000714_0001
 - 3000714_0004
 - 3000714_0009
 - 3000714_0010
 - 3000714_0012
 - 3000714_0014
 - 3000714_0016
 - 3000714_0018
 - 3000714_0020
 - 3000714_0024
 - 3000714_0032
 - 3000714_0035
 - 3000714_0038
 - 3000717_0002
 - 3000717_0006
 - 3000717_0016
 - 3000717_0021
 - 3000717_0023
 - 3000717_0024
 - 3000717_0033
 - 3000717_0036
 - 3000717_0040
 - 3000717_0041
 - 3000717_0042
 - 3000717_0046
 - 3000717_0053
 - 3000717_0059
 - 3000717_0067
 - 3000717_0068
 - 3000717_0069
 - 3000717_0072
 - 3000717_0077
 - 3000717_0078
 - 3000781_0003
 - 3000781_0004
 - 3000781_0005
 - 3000847_0011
 - 3000847_0014
 - 3000847_0018
 - 3000847_0020
 - 3000847_0025
 - 3000847_0027
 - 3000847_0030
 - 3000847_0031
 - 3000847_0032
 - 3000847_0033
 - 3000847_0035
 - 3000847_0036
 - 3000847_0038
 - 3000847_0039
 - 3000847_0040
 - 3000847_0042
 - 3000847_0047
 - 3000847_0048
 - 3000847_0049
 - 3000847_0050
 - 3000847_005

In [51]:
import os
FILE_RECORDS = "matching_records.csv"
if matching_recs:
    print("\nConsolidando, removendo duplicatas e ordenando o CSV Geral...")
    arquivo_geral = FILE_RECORDS
    
    # Dados novos gerados nesta execução
    df_new = pd.DataFrame(matching_recs, columns=['dir', 'seg_name', 'length'])
    
    if os.path.isfile(arquivo_geral):
        print("File is in os.path")
        df_existing = pd.read_csv(arquivo_geral)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
        
    # Garante a atualização substituindo duplicatas antigas e ordena
    df_combined.drop_duplicates(subset=['seg_name'], keep='last', inplace=True)
    df_combined.sort_values(by=['seg_name', 'dir'], inplace=True)
    
    # Gravação final única em disco
    print(f"Head: {df_combined.head()} End: {df_combined[-5:]}")
    df_combined.to_csv(arquivo_geral, index=False)
    print(f"Processo concluído! Arquivo final salvo em: {arquivo_geral}")
else:
    print("\n❌ Nenhum flush detectado nos casos analisados.")


Consolidando, removendo duplicatas e ordenando o CSV Geral...
File is in os.path
Head:                         dir      seg_name     length  Case_ID  Image_Name  \
0  mimic3wdb/1.0/30/3000003  3000003_0014  19200.000      NaN         NaN   
1  mimic3wdb/1.0/30/3000063  3000063_0033  10990.000      NaN         NaN   
2  mimic3wdb/1.0/30/3000105  3000105_0011   1997.184      NaN         NaN   
3  mimic3wdb/1.0/30/3000126  3000126_0013  65100.000      NaN         NaN   
4  mimic3wdb/1.0/30/3000189  3000189_0022  31560.000      NaN         NaN   

   Start_Pos  
0        NaN  
1        NaN  
2        NaN  
3        NaN  
4        NaN   End:                           dir      seg_name   length  Case_ID  Image_Name  \
596  mimic3wdb/1.0/30/3003521  3003521_0026    466.0      NaN         NaN   
597  mimic3wdb/1.0/30/3003521  3003521_0027  15675.0      NaN         NaN   
598  mimic3wdb/1.0/30/3003521  3003521_0030  18863.0      NaN         NaN   
599  mimic3wdb/1.0/30/3003521  3003521_0032   

# Geração dos gráficos

In [2]:
import os
import vitaldb
import wfdb  
import matplotlib
matplotlib.use('Tkagg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

c:\Users\heito\anaconda3\envs\tf_gpu\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
records_csv = pd.read_csv("matching_records.csv")
idx_record = 17
segment_graph = records_csv.iloc[idx_record]
print(segment_graph)

Unnamed: 0                          17
dir           mimic3wdb/1.0/30/3000063
seg_name                  3000063_0020
length                       38190.736
Name: 17, dtype: object


In [6]:
# Master_recors
seg_name = segment_graph.seg_name
seg_dir = segment_graph.dir
# DF
# segment_names = df.iloc[1].seg_name
# segment_dirs = df.iloc[1].dir

print(f"Segment name: {seg_name} Segment dir: {seg_dir}")

Segment name: 3000063_0020 Segment dir: mimic3wdb/1.0/30/3000063


In [7]:
from pprint import pprint

# O atributo segments serve para armazenar os objetos de cabeçalho individuais de cada um desses
# 23 segmentos separados. Como baixar e ler 23 arquivos .hea adicionais da rede pode ser um processo 
# lento, a biblioteca não faz isso automaticamente. Por isso, o atributo segments retorna None. Para 
# que a biblioteca leia os cabeçalhos de todos os segmentos individuais e preencha a lista do atributo 
# segments, você precisa passar o argumento explícito rd_segments=True para a função.
record__name = seg_name.split("_")[0]
record_data = wfdb.rdheader(record_name=record__name, pn_dir=seg_dir, rd_segments=True) 
print("------------RECORD METADATA LOADED---------------")
pprint(vars(record_data))


segment_data = wfdb.rdheader(record_name=seg_name, pn_dir=seg_dir) 
print("\n---------SEGMENT METADATA LOADED--------------")
pprint(vars(segment_data))

------------RECORD METADATA LOADED---------------
{'base_counter': None,
 'base_date': None,
 'base_time': datetime.time(19, 57, 4, 360000),
 'comments': ['Location: tsicu'],
 'counter_freq': None,
 'fs': 125,
 'layout': 'variable',
 'n_seg': 38,
 'n_sig': 4,
 'record_name': '3000063',
 'seg_len': [0,
             7830,
             250,
             750,
             125,
             1875,
             250,
             6375,
             589375,
             742015,
             382110,
             1750,
             34568,
             89344,
             357504,
             74624,
             120448,
             276224,
             277376,
             1001537,
             2750,
             604033,
             112000,
             4773842,
             10000,
             5847070,
             1417305,
             66375,
             2395875,
             7875,
             6875,
             12663,
             8849712,
             43125,
             46299,
           

In [8]:
print(f"Data stored in class of type for record: {type(record_data)}")
print(f"Data stored in class of type for segment: {type(segment_data)}")

Data stored in class of type for record: <class 'wfdb.io.record.MultiRecord'>
Data stored in class of type for segment: <class 'wfdb.io.record.Record'>


In [9]:
print(f"This segment contains waveform data for the following {segment_data.n_sig} signals: {segment_data.sig_name}")
print(f"The signals are sampled at a base rate of {segment_data.fs} Hz (and some are sampled at multiples of this)")
print(f"They last for {segment_data.sig_len/(60*segment_data.fs):.1f} minutes")

This segment contains waveform data for the following 3 signals: ['II', 'ABP', 'PLETH']
The signals are sampled at a base rate of 125 Hz (and some are sampled at multiples of this)
They last for 636.5 minutes


In [10]:
fs = segment_data.fs
length = segment_data.sig_len
sampfrom = 0

segment_data = wfdb.rdrecord(record_name=seg_name,
                             sampfrom=sampfrom,
                             pn_dir=seg_dir)

print(f"{round(length/fs)} seconds of data extracted from segment {seg_name}")

38191 seconds of data extracted from segment 3000063_0020


In [11]:
title_text = f"Segment {seg_name}"
wfdb.plot_wfdb(record=segment_data,
               title=title_text,
               time_units='seconds') 

In [12]:
for sig_no in range(0, len(segment_data.sig_name)):
    if "ABP" in segment_data.sig_name[sig_no]:
        break

abp = segment_data.p_signal[:, sig_no]
print(f"Extracted the ABP signal from column {sig_no} of the matrix of waveform data.")

Extracted the ABP signal from column 1 of the matrix of waveform data.


In [13]:
from matplotlib import pyplot as plt
import numpy as np

abp_clean = abp[~np.isnan(abp)]
t = np.arange(0, (len(abp_clean) / fs), 1.0 / fs)
plt.plot(t, abp_clean, color = 'black', label='ABP')
plt.show()

In [15]:
SAMPLE_RATE = fs
THRESHOLD_HIGH = 120
THRESHOLD_LOW = 110
DIFF_THRESHHOLD = 180 / (0.5 * SAMPLE_RATE)

def _process_signal(signal_raw, t, case_label, file_prefix, dir_images, dir_csvs):
    """Detecta flushes em milissegundos e salva imagens/csvs rapidamente."""
    signal_raw_filtered = signal_raw[~np.isnan(signal_raw)]
    if len(signal_raw_filtered) == 0:
        return []

    time = t
    sample_rate = fs
    
    # Adicionando .astype(int) para transformar True/False em 1/0
    is_high = (signal_raw_filtered > THRESHOLD_HIGH).astype(int)
    is_low = (signal_raw_filtered < THRESHOLD_LOW).astype(int)

    # Agora a subtração matemática (1 - 0 = 1, ou 0 - 1 = -1) funcionará perfeitamente
    high_indices = np.where(np.diff(np.concatenate(([0], is_high))) >= 1)[0]
    low_indices = np.where(np.diff(np.concatenate(([0], is_low))) >= 1)[0]


    print(f"First flush: \n Is high: {is_high} \n Is low: {is_low}")

    print(f"Index: \n High indices: {high_indices} Low indices: {low_indices}")
    print(f"Time instant: \n High indices: {np.round(high_indices/fs, 2)} Low indices: {np.round(low_indices/fs, 2)}")

    flushes_encontrados = []
    last_des = -1

    for sub in high_indices:
        if sub <= last_des: continue  
        
        idx_low = np.searchsorted(low_indices, sub)
        if idx_low < len(low_indices):
            des = low_indices[idx_low]
            if 1 < ((des - sub) / sample_rate) < 19:
                flushes_encontrados.append([sub, des])
            else:
                print(f"Não satisfaz condição de duração: {round(sub/fs)} até {round(des/fs)}")
            last_des = des
        else:
            print("Súbida após última descida")
            break  

    if not flushes_encontrados:
        return [],[]

    # 2. Geração de Gráficos e Arquivos
    resultados = []

    os.makedirs(dir_images, exist_ok=True)
    os.makedirs(dir_csvs, exist_ok=True)
    
    # Cria uma única figura para o caso inteiro
    fig, ax = plt.subplots(1, 1, layout='constrained', figsize=(12, 8))

    for idx, (sub, des) in enumerate(flushes_encontrados):
        start_pos = max(0, des - 20 * sample_rate)
        end_pos = min(len(signal_raw_filtered), des + 20 * sample_rate)
        
        # Limpa o eixo para o próximo gráfico (economiza muita RAM)
        ax.clear()
        
        t_seg = time[start_pos:end_pos]
        sinal_recorte = signal_raw_filtered[start_pos:end_pos]
        
        ax.plot(t_seg, sinal_recorte, label="sinal", color='#1f77b4')
        ax.axvline(time[sub], color='red', linestyle='--', label='Início Flush')
        ax.axvline(time[des], color='red', linestyle='-', label='Fim Flush')
        ax.axhline(THRESHOLD_HIGH, xmin=0.1, xmax=0.9, color='red', alpha=0.5, label='Threshold')
        ax.set_title(f'Flush {idx} - Caso {case_label}')
        ax.set_ylabel('Amplitude')
        ax.legend(loc='upper right')
        ax.grid(True)

        name_file = f'{file_prefix}_flush_{idx}'
        
        # Exporta CSV super rápido
        save_flush_csv = os.path.join(dir_csvs, f"{name_file}.csv")
        t_centralizado = np.arange(-20 * sample_rate, len(sinal_recorte) - 20 * sample_rate)
        try:
            data_export = np.column_stack((t_centralizado, sinal_recorte))
            np.savetxt(save_flush_csv, data_export, delimiter=",", fmt="%.4f")
        except ValueError:
            continue

        # Exporta a Imagem
        save_flush_img = os.path.join(dir_images, f"{name_file}.png")
        fig.savefig(save_flush_img, bbox_inches='tight')

        resultados.append([case_label, name_file, (sub - des)])

    plt.close(fig)
    print(f"-> [Sucesso] Caso {case_label}: Processado. {len(flushes_encontrados)} flushes exportados.")
    return resultados, flushes_encontrados

resultados, flushes_encontrados = _process_signal(abp, t, "teste", "mimic_teste", "mimic_test_img", "mimic_test_csv")
flushes_encontrados = np.round(np.array(flushes_encontrados)/fs, 2)
print(resultados, flushes_encontrados)


First flush: 
 Is high: [0 0 0 ... 0 0 0] 
 Is low: [1 1 1 ... 1 1 1]
Index: 
 High indices: [     35  530443  532210 ... 4739444 4739757 4740016] Low indices: [      0      48     120 ... 4740708 4740761 4740811]
Time instant: 
 High indices: [2.800000e-01 4.243540e+03 4.257680e+03 ... 3.791555e+04 3.791806e+04
 3.792013e+04] Low indices: [0.000000e+00 3.800000e-01 9.600000e-01 ... 3.792566e+04 3.792609e+04
 3.792649e+04]
Não satisfaz condição de duração: 0 até 0
Não satisfaz condição de duração: 4244 até 4244
Não satisfaz condição de duração: 4258 até 4258
Não satisfaz condição de duração: 4258 até 4258
Não satisfaz condição de duração: 4259 até 4259
Não satisfaz condição de duração: 4262 até 4262
Não satisfaz condição de duração: 4263 até 4263
Não satisfaz condição de duração: 4264 até 4264
Não satisfaz condição de duração: 4264 até 4264
Não satisfaz condição de duração: 4265 até 4265
Não satisfaz condição de duração: 4267 até 4267
Não satisfaz condição de duração: 4269 até 4269
Não